# Predict

Run prediction using the model output from [02_modelling.ipynb](./02_modelling.ipynb)

Output: [nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson](https://storage.googleapis.com/niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson)

Comparison map [here](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22__User-Added_Data__%22%3A%7B%22isOpen%22%3Atrue%2C%22members%22%3A%5B%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%5D%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%3A%7B%22splitDirection%22%3A1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%3A%7B%22splitDirection%22%3A-1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%3A%7B%22splitSourceItemId%22%3A%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22dereferenced%22%3A%7B%22name%22%3A%22Test+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge+%28copy%29%22%2C%22splitDirection%22%3A-1%7D%2C%22knownContainerUniqueIds%22%3A%5B%22__User-Added_Data__%22%5D%2C%22type%22%3A%22split-reference%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%5D%2C%22timeline%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A-1.8237304687500002%2C%22south%22%3A60.31062731740045%2C%22east%22%3A18.665771484375004%2C%22north%22%3A65.29346780107583%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Atrue%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [ ]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import xdem
from osgeo import gdal, ogr, osr
import rasterstats


import subkart

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

# Predict for Norge

## Wave Exposure


In [4]:
bolge = subkart.sources.bolge_exposure()


## DEM 50

In [5]:
dem_norge = subkart.sources.dem_data()

## Predict

In [ ]:
def to_raster(name, pred_array, valid_attrs, out_shape, transform, nodata=nodata):
    pred_map = np.full(out_shape, nodata, dtype=np.uint8)
    pred_map[valid_attrs] = pred_array
    pred_map_masked = np.ma.masked_equal(pred_map, nodata)
    pred_raster = gu.Raster.from_array(pred_map_masked, transform=transform, crs=crs, nodata=nodata)
    pred_raster.to_file(Path(f"{name}.tif"))


In [ ]:
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}

for region_name, region_list in subkart.sources.REGIONS.items():
    print(f"Processing region: {region_name}")
    gdf_sea_map_region = subkart.sources.sea_map_basisdata(region_list)
    gdf_sea_map_region = subkart.features.depth_preprocess(gdf_sea_map_region)
    transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map_region, res=res)
    dem_region = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)
    print("Preparing bolge_region")
    bolge_region = bolge.reproject(
        crs=crs, res=res, bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3])
    )
    print("Preparing X, valid_attrs, out_shape, transform")
    X, valid_attrs, out_shape, transform = subkart.features.build(
        dem_region, gdf_sea_map_region, bolge_region, valid_mask=None, res=res, dtype=np.float32
    )
    print("Predict ...")
    Y_pred = classifier.predict(X)
    Y_prob = classifier.predict_proba(X)

    # 1. Prediction raster (classes 0=løsbunn, 1=blanding, 2=fastbunn)
    to_raster(f"{region_name}_prediction", Y_pred.astype(np.uint8), valid_attrs, out_shape, transform)
    print(f"Saved prediction for region {region_name}")

    # 2. Three-band probability raster (band 1=P(løsbunn), band 2=P(blanding), band 3=P(fastbunn))
    prob_nodata = np.float32(-9999)
    prob_bands = np.full((len(classifier.classes_), *out_shape), prob_nodata, dtype=np.float32)
    for band_idx, cls in enumerate(classifier.classes_):
        prob_bands[band_idx][valid_attrs] = Y_prob[:, band_idx].astype(np.float32)
    prob_fname = f"{region_name}_probability.tif"
    with rio.open(
        prob_fname, "w",
        driver="GTiff",
        height=out_shape[0], width=out_shape[1],
        count=len(classifier.classes_),
        dtype=np.float32,
        crs=crs,
        transform=transform,
        nodata=prob_nodata,
    ) as dst:
        dst.write(prob_bands)
    print(f"Saved probability raster for region {region_name}\n")


Processing region: vestland


Preparing bolge_region


Preparing X, valid_attrs, out_shape, transform
Preparing sea_avg_depth...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_avg_slope...


...

/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...


Stacking feature arrays...


Predict ...


Saved prediction for region nord to nord_prediction.tif



## Post processing

In [ ]:
gdal.UseExceptions()

fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"
prob_file = f"{fname}_probability.tif"

prediction_files = [f"{r}_prediction.tif" for r in subkart.sources.REGIONS]
probability_files = [f"{r}_probability.tif" for r in subkart.sources.REGIONS]

subkart.utils.merge_rasters(prediction_files, predict_file, nodata=nodata)
subkart.utils.merge_rasters(probability_files, prob_file, nodata=np.float32(-9999))


In [ ]:
# Vectorize raw prediction and write final raw nisjedata-substrat vector
subkart.vectorize.with_gdal(predict_file, "polygons_raw.gpkg", int(crs.split(":")[1]), nodata=255)

gdf_raw = gpd.read_file("polygons_raw.gpkg")
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf_raw["BunnType"] = gdf_raw["DN"].map(reverse_map)

fname_raw = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}-raw",
    "norge", "latest", gdf_raw.crs.to_epsg(),
)
gdf_raw.to_file(f"{fname_raw}.gpkg", driver="GPKG", layer="bunntyper")
gdf_raw.to_parquet(f"{fname_raw}.geo.parquet", compression="snappy")
subkart.utils.to_postgis(gdf_raw, fname_raw)


In [ ]:
# Create processed prediction raster:
#   1. Remap class 1 (blanding) → 0 (løsbunn) using gdal_calc
#   2. Sieve filter: replace isolated single pixels with their largest neighbour
from osgeo_utils import gdal_calc

predict_file_remapped = f"{fname}_remapped_tmp.tif"
predict_file_processed = f"{fname}_processed.tif"

gdal_calc.Calc(
    calc="numpy.where(A==1, numpy.uint8(0), A)",
    outfile=predict_file_remapped,
    A=predict_file,
    type="Byte",
    NoDataValue=nodata,
    hideNoData=True,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)

# Pre-fill processed file with remapped data; SieveFilter only writes changed pixels
gdal.Translate(
    predict_file_processed,
    predict_file_remapped,
    creationOptions=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
)

ds_src = gdal.Open(predict_file_remapped, gdal.GA_ReadOnly)
ds_dst = gdal.Open(predict_file_processed, gdal.GA_Update)
src_band = ds_src.GetRasterBand(1)
dst_band = ds_dst.GetRasterBand(1)
gdal.SieveFilter(
    src_band,
    src_band.GetMaskBand(),
    dst_band,
    threshold=1,
    connectedness=8,
    callback=gdal.TermProgress_nocb,
)
ds_src = None
ds_dst = None

Path(predict_file_remapped).unlink()


In [ ]:
# Create processed 1-band probability raster from the 3-band source:
#   class 0 (løsbunn, incl. remapped/sieved blanding): band 1 = P(class=0)
#   class 2 (fastbunn):                                 band 3 = P(class=2)
prob_file_processed = f"{fname}_probability_processed.tif"

gdal_calc.Calc(
    calc="numpy.where(A==2, C, B)",
    outfile=prob_file_processed,
    A=predict_file_processed,
    B=prob_file,
    B_band=1,
    C=prob_file,
    C_band=3,
    type="Float32",
    NoDataValue=-9999,
    hideNoData=True,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)


In [ ]:

# Vectorize processed prediction raster
subkart.vectorize.with_gdal(
    predict_file_processed, "polygons_processed.gpkg", int(crs.split(":")[1]), nodata=255
)

gdf = gpd.read_file("polygons_processed.gpkg")
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-9999,
)
gdf["probability"] = [s["mean"] for s in stats]

# Douglas-Peucker simplification (25 m threshold collapses raster staircase steps)
gdf["geometry"] = gdf.geometry.simplify(
    tolerance=subkart.vectorize.SMOOTH_THRESHOLD, preserve_topology=True
)
gdf = gdf[~gdf.geometry.is_empty].reset_index(drop=True)

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")
subkart.utils.to_postgis(gdf, fname)
